# 📃 Solução do Exercício M1.04

O objetivo deste exercício é avaliar o impacto de usar uma codificação
arbitrária em números inteiros para variáveis categóricas junto com um modelo
de classificação linear, como a Regressão Logística.

Para isso, vamos tentar usar o `OrdinalEncoder` para pré-processar as variáveis
categóricas. Esse pré-processador é montado em um pipeline com
`LogisticRegression`. O desempenho de generalização do pipeline pode ser
avaliado por validação cruzada e depois comparado com o score obtido ao usar
`OneHotEncoder` ou com algum outro score de referência.

Primeiro, carregamos o conjunto de dados.

In [ ]:
import pandas as pd

adult_census = pd.read_csv("../datasets/adult-census.csv")

In [ ]:
target_name = "class"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name, "education-num"])

No notebook anterior, usamos `sklearn.compose.make_column_selector` para
selecionar automaticamente colunas com um tipo de dado específico (também
chamado de `dtype`). Aqui, usamos esse seletor para obter apenas as colunas que
contêm strings (coluna com `dtype` `object`) que correspondem às features
categóricas em nosso conjunto de dados.

In [ ]:
from sklearn.compose import make_column_selector as selector

categorical_columns_selector = selector(dtype_include=object)
categorical_columns = categorical_columns_selector(data)
data_categorical = data[categorical_columns]

Defina um pipeline do scikit-learn composto por um `OrdinalEncoder` e um
classificador `LogisticRegression`.

Como o `OrdinalEncoder` pode gerar erros se encontrar uma categoria
desconhecida no momento da previsão, você pode definir os parâmetros
`handle_unknown="use_encoded_value"` e `unknown_value`. Você pode consultar a
[documentação do
scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html)
para mais detalhes sobre esses parâmetros.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression

# solução
model = make_pipeline(
    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
    LogisticRegression(max_iter=500),
)

Seu modelo já está definido. Avalie-o usando validação cruzada com
`sklearn.model_selection.cross_validate`.

<div class="admonition note alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Nota</p>
<p class="last">Saiba que, se ocorrer um erro durante a validação cruzada, o
<tt class="docutils literal">cross_validate</tt> emitirá um aviso e retornará
NaN (Not a Number) como scores. Para fazer com que ele gere uma exceção
Python padrão com um traceback, você pode passar o argumento
<tt class="docutils literal"><span class="pre">error_score="raise"</span></tt>
na chamada de <tt class="docutils literal">cross_validate</tt>. Uma exceção
seria gerada em vez de um aviso assim que o primeiro problema fosse
encontrado, e o <tt class="docutils literal">cross_validate</tt> pararia
imediatamente em vez de retornar valores NaN. Isso é particularmente útil ao
desenvolver pipelines de aprendizado de máquina complexos.</p>
</div>

In [ ]:
from sklearn.model_selection import cross_validate

# solução
cv_results = cross_validate(model, data_categorical, target)

scores = cv_results["test_score"]
print(
    "A acurácia média da validação cruzada é: "
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

Usar um mapeamento arbitrário de rótulos de texto para inteiros, como feito
aqui, faz com que o modelo linear assuma suposições ruins sobre a ordenação
relativa das categorias.

Isso impede que o modelo aprenda algo suficientemente preditivo, e o score
obtido por validação cruzada é ainda menor do que a referência que obtivemos
ignorando os dados de entrada e apenas prevendo constantemente a classe mais
frequente:

In [ ]:
from sklearn.dummy import DummyClassifier

cv_results = cross_validate(
    DummyClassifier(strategy="most_frequent"), data_categorical, target
)
scores = cv_results["test_score"]
print(
    "A acurácia média da validação cruzada é: "
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

Agora, gostaríamos de comparar o desempenho de generalização do nosso modelo
anterior com um novo modelo em que, em vez de usar um `OrdinalEncoder`, usamos
um `OneHotEncoder`. Repita a avaliação do modelo usando validação cruzada.
Compare o score dos dois modelos e conclua sobre o impacto de escolher uma
estratégia de codificação específica ao usar um modelo linear.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# solução
model = make_pipeline(
    OneHotEncoder(handle_unknown="ignore"), LogisticRegression(max_iter=500)
)
cv_results = cross_validate(model, data_categorical, target)
scores = cv_results["test_score"]
print(
    "A acurácia média da validação cruzada é: "
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

Com o classificador linear escolhido, usar uma codificação que não assume
nenhuma ordenação levou a um resultado muito melhor.

A mensagem importante aqui é: modelo linear e `OrdinalEncoder` são usados
juntos apenas para features categóricas ordinais, ou seja, features que têm uma
ordenação específica. Caso contrário, seu modelo teria um desempenho ruim.